In [1]:
import os 
from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool
from tavily import TavilyClient
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.graph.message import add_messages

In [2]:
load_dotenv()

True

In [3]:
llm = init_chat_model(model="groq:openai/gpt-oss-20b")

In [4]:
#defining state
class AgentState(MessagesState):
    next_agent: str

In [5]:
@tool
def web_search(query: str, **kwargs) -> str:
    """search the web for information"""
    client = TavilyClient()
    results = client.search(query=query, max_results=3)
    snippets = []
    for r in results.get("results", []):
        snippets.append(f"Title: {r.get('title')}\nURL: {r.get('url')}\nContent: {r.get('content')}")
    return "\n\n---\n\n".join(snippets) if snippets else "No results found."

In [6]:
#defining agent function
def researcher_agent(state:AgentState):
    """researcher agent that searches for information"""

    user_query = state['messages'][-1].content

    # use LLM to generate a focused search query
    query_response = llm.invoke([SystemMessage(content="Convert the user's question into a concise, keyword-rich web search query. Return ONLY the search query, nothing else."), HumanMessage(content=user_query)])
    search_query = query_response.content.strip().strip('"')

    search_results = web_search.invoke({"query": search_query})

    return {
        "messages": [AIMessage(content=f"Search results for '{search_query}':\n\n{search_results}")],
        "next_agent": "writer"
    }


In [7]:
def writer_agent(state: AgentState):
    """writer agent that summarizes content""" 

    system_msg = SystemMessage(content = "you are a professional writer who reviews the content and then summarizes it for proper format and professional way")

    response = llm.invoke([system_msg]+ state['messages'])

    return {
        "messages" : [response],
        "next_agent" : "end"
    }

In [8]:
workflow = StateGraph(AgentState)

# adding nodes
workflow.add_node("researcher", researcher_agent)
workflow.add_node("writer", writer_agent)

In [9]:
# defining flow/edges
workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

final_workflow = workflow.compile()

In [11]:
query = "can you research and tell me about father of machine learning?"

result = final_workflow.invoke({"messages": [HumanMessage(content=query)]})

messages = result["messages"]

# Pretty print Research Output (from researcher agent)
print("=" * 60)
print("📚 RESEARCH OUTPUT")
print("=" * 60)
for msg in messages:
    if isinstance(msg, AIMessage) and msg.content.startswith("Search results"):
        print(msg.content)
        break

# Pretty print Summarized Output (from writer agent)
print("\n")
print("=" * 60)
print("✍️  SUMMARIZED OUTPUT")
print("=" * 60)
print(messages[-1].content)

📚 RESEARCH OUTPUT
Search results for 'father of machine learning':

Title: Father of Machine Learning and Artificial Intelligence - Shef Solutions LLC
URL: https://shefsolutionsllc.com/father-of-machine-learning-and-artificial-intelligence
Content: While John McCarthy established the foundation of AI, it was Arthur Samuel who brought the concept of learning into the field. Samuel is widely recognized as the Father of Machine Learning because of his pioneering work in developing programs that could improve their performance through experience. [...] of Artificial Intelligence, and Arthur Samuel, regarded as the Father of Machine Learning. [...] ## John McCarthy – The Father of Artificial Intelligence

John McCarthy, an American computer scientist, is celebrated as the Father of Artificial Intelligence. In 1956, he organized the famous Dartmouth Conference and coined the term Artificial Intelligence. This conference brought together some of the greatest minds in computer science and mark